# Merge experimental literature samples into main.csv

This notebook adds new literature samples (from `experimental_data_samples.xlsx`) to the
existing `main.csv` dataset **without modifying `main.csv` itself**. The result is written
to a new file, `main_1.csv`.

## Data-quality issue found in the source spreadsheet

Several numeric cells in `experimental_data_samples.xlsx` were auto-converted into **dates**
by Excel (e.g. a typed value of `14.4` in a `day.month`-formatted cell became "April 14").
This was recoverable: the cell's `number_format` (`d.m`, `dd.mm`, or `m.yyyy`) tells us exactly how the
day/month/year should be reassembled back into the original decimal number. Each recovered value below
is flagged with a comment and was sanity-checked against the physical quantity it represents
(e.g. TiC lattice parameter ~4.3 Å, Dv50 particle size ~10-15 µm, wt% additions of a few percent).

## Samples excluded

The source file contains 34 candidate samples across 6 literature blocks. **4 samples were
excluded** (V6L-Mm-AC, V6L-Mm-4h, V6.5L-Mm-AC, V6.5L-Mm-8h) because they use "Mm" (mischmetal,
an undefined mixture of rare-earth elements) as an additive, and `main.csv` has no column for
an unspecified rare-earth mix and the paper note doesn't give the exact La/Ce/Nd/Pr ratio used.
This leaves **30 new samples** to merge.

## Unit handling

`main.csv`'s element columns are **atomic fractions that sum to 1** per row (verified below),
not weight percent. The literature blocks report composition in different conventions:

- **wt% composition** (block 1, block 5): converted to atomic fraction via standard atomic weights.
- **stoichiometric/formula subscripts or already-atomic%** (blocks 2, 3, 4, 6): converted to atomic
  fraction by dividing by the row sum (this is equivalent to formula normalization, and also
  correctly handles atomic-% rows where dopants are reported as "extra" atomic percent on top of
  a non-100 base).

No plateau pressures (`P_abs`/`P_des`) were fabricated from enthalpy (ΔH) data — where a paper reports
only ΔH/ΔS (van 't Hoff thermodynamics) rather than a measured plateau pressure, `P_abs`/`P_des` are
left as `NaN`, consistent with how `main.csv` already tolerates missing pressure values.


In [1]:
import pandas as pd
import numpy as np

main = pd.read_csv('main.csv')
element_cols = main.columns[4:].tolist()
print(main.shape)
assert list(main.columns[:4]) == ['table', 'wt', 'P_abs', 'P_des']
print('element columns:', len(element_cols))
print('row sums (should all be 1):', main[element_cols].sum(axis=1).describe()[['min', 'max']].to_dict())


(179, 62)
element columns: 58
row sums (should all be 1): {'min': 0.9999999999999996, 'max': 1.000001}


## Atomic weights

Used only for the two blocks reported in wt% (block 1 and block 5). Blocks reported as
stoichiometric formula subscripts or already in atomic % don't need this — they're normalized
by their own row sum directly.


In [2]:
ATOMIC_WEIGHT = {
    'Ti': 47.867, 'Fe': 55.845, 'Zr': 91.224, 'Y': 88.906, 'Ni': 58.6934,
    'Mn': 54.938, 'V': 50.9415, 'C': 12.011, 'Ce': 140.116, 'Cr': 51.996,
    'Co': 58.933, 'Cu': 63.546, 'S': 32.06,
}

def wt_pct_to_atomic_fraction(wt_pct: dict) -> dict:
    """Convert a dict of {element: wt%} to {element: atomic fraction}, normalized to sum to 1."""
    moles = {el: w / ATOMIC_WEIGHT[el] for el, w in wt_pct.items() if w}
    total = sum(moles.values())
    return {el: m / total for el, m in moles.items()}

def ratio_to_atomic_fraction(ratio: dict) -> dict:
    """Convert a dict of {element: stoichiometric coefficient or at%} to atomic fraction, normalized to sum to 1."""
    nonzero = {el: v for el, v in ratio.items() if v}
    total = sum(nonzero.values())
    return {el: v / total for el, v in nonzero.items()}

def make_row(table_id, elements_atomic_fraction, wt, p_abs=np.nan, p_des=np.nan):
    row = {col: 0.0 for col in element_cols}
    for el, frac in elements_atomic_fraction.items():
        row[el] = frac
    row['table'] = table_id
    row['wt'] = wt
    row['P_abs'] = p_abs
    row['P_des'] = p_des
    return row


### Block 1 — Y–TiFe + V composites (5 samples)

Paper: *Effect on the activation and hydrogen storage properties of Y–TiFe-based composites
with vanadium via mechanical milling*. Elemental wt% already computed by hand from the base
alloy formula Ti₁.₀₈Zr₀.₁Y₀.₀₂Fe₀.₆Ni₀.₃Mn₀.₂ scaled by added V content (per the sheet's own note).
No plateau pressure reported.


In [3]:
block1_raw = [
    dict(Ti=41.457, Fe=26.870, Zr=7.316, Y=1.426, Ni=14.120, Mn=8.811, V=0.0, wt=1.704),
    dict(Ti=40.628, Fe=26.333, Zr=7.169, Y=1.397, Ni=13.838, Mn=8.635, V=2.0, wt=1.715),
    dict(Ti=39.384, Fe=25.527, Zr=6.950, Y=1.355, Ni=13.414, Mn=8.371, V=5.0, wt=1.732),
    dict(Ti=38.555, Fe=24.989, Zr=6.803, Y=1.326, Ni=13.132, Mn=8.194, V=7.0, wt=1.730),
    dict(Ti=37.311, Fe=24.183, Zr=6.584, Y=1.283, Ni=12.708, Mn=7.930, V=10.0, wt=1.620),
]

block1_rows = []
for i, s in enumerate(block1_raw, start=1):
    wt_pct = {el: s[el] for el in ('Ti', 'Fe', 'Zr', 'Y', 'Ni', 'Mn', 'V')}
    frac = wt_pct_to_atomic_fraction(wt_pct)
    block1_rows.append(make_row(f'4_{i}', frac, s['wt']))

len(block1_rows)


5

### Block 2 — TiFe0.7Mn0.2X0.1 series (6 samples)

Paper: *Microstructural feature and hydrogen storage properties of TiFe0.7Mn0.2X0.1
(X = V, Cr, Co, Ni, Cu) hydrogen storage alloy*. Values are stoichiometric formula subscripts.
Only ΔH/ΔS (thermodynamics) are reported, not a measured plateau pressure, so `P_abs`/`P_des`
are left as `NaN`.


In [4]:
block2_raw = [
    dict(Ti=1.0, Fe=0.7, Mn=0.2, V=0.0, Cr=0.0, Co=0.0, Ni=0.0, Cu=0.0, wt=1.77),
    dict(Ti=1.0, Fe=0.7, Mn=0.2, V=0.1, Cr=0.0, Co=0.0, Ni=0.0, Cu=0.0, wt=1.71),
    dict(Ti=1.0, Fe=0.7, Mn=0.2, V=0.0, Cr=0.1, Co=0.0, Ni=0.0, Cu=0.0, wt=1.79),
    dict(Ti=1.0, Fe=0.7, Mn=0.2, V=0.0, Cr=0.0, Co=0.1, Ni=0.0, Cu=0.0, wt=1.67),
    dict(Ti=1.0, Fe=0.7, Mn=0.2, V=0.0, Cr=0.0, Co=0.0, Ni=0.1, Cu=0.0, wt=1.36),
    dict(Ti=1.0, Fe=0.7, Mn=0.2, V=0.0, Cr=0.0, Co=0.0, Ni=0.0, Cu=0.1, wt=1.56),
]

block2_rows = []
start = len(block1_rows) + 1
for i, s in enumerate(block2_raw, start=start):
    ratio = {el: s[el] for el in ('Ti', 'Fe', 'Mn', 'V', 'Cr', 'Co', 'Ni', 'Cu')}
    frac = ratio_to_atomic_fraction(ratio)
    block2_rows.append(make_row(f'4_{i}', frac, s['wt']))

len(block2_rows)


6

### Block 3 — TiFe–V (+Ce) alloys (12 of 16 samples; 4 excluded)

Paper: *Optimizing hydrogen storage in TiFe–V alloys: Influence of sample purity and heat
treatment*. Sample names encode composition directly in atomic % (e.g. "V5" = 5 at% V).
`V6L-AC`'s capacity (`H29`) was one of the corrupted date cells — recovered as `1.1` (consistent
with the 1.2-1.5 wt% range of its neighbors). The four `*-Mm-*` samples are excluded (see intro).
No plateau pressure reported for this block.


In [5]:
block3_raw = [
    dict(name='V5H-AC',      Ti=48, Fe=47, V=5, Ce=0, wt=1.23),
    dict(name='V5H-4h',      Ti=48, Fe=47, V=5, Ce=0, wt=1.52),
    dict(name='V5L-AC',      Ti=48, Fe=47, V=5, Ce=0, wt=1.20),
    dict(name='V5L-4h',      Ti=48, Fe=47, V=5, Ce=0, wt=1.31),
    dict(name='V5L-Ce-AC',   Ti=48, Fe=47, V=5, Ce=3, wt=1.25),
    dict(name='V5L-Ce-4h',   Ti=48, Fe=47, V=5, Ce=3, wt=1.38),
    dict(name='V6H-AC',      Ti=47, Fe=47, V=6, Ce=0, wt=1.32),
    dict(name='V6H-4h',      Ti=47, Fe=47, V=6, Ce=0, wt=1.54),
    dict(name='V6L-AC',      Ti=47, Fe=47, V=6, Ce=0, wt=1.1),   # recovered from corrupted date cell H29
    dict(name='V6L-4h',      Ti=47, Fe=47, V=6, Ce=0, wt=1.22),
    dict(name='V6L-Ce-AC',   Ti=47, Fe=47, V=6, Ce=3, wt=1.28),
    dict(name='V6L-Ce-4h',   Ti=47, Fe=47, V=6, Ce=3, wt=1.43),
    # excluded: V6L-Mm-AC, V6L-Mm-4h, V6.5L-Mm-AC, V6.5L-Mm-8h (undefined mischmetal composition)
]

block3_rows = []
start = len(block1_rows) + len(block2_rows) + 1
for i, s in enumerate(block3_raw, start=start):
    ratio = {el: s[el] for el in ('Ti', 'Fe', 'V', 'Ce')}
    frac = ratio_to_atomic_fraction(ratio)
    block3_rows.append(make_row(f'4_{i}', frac, s['wt']))

len(block3_rows)


12

### Block 4 — rare-earth-doped TiFe under cycling (1 sample)

Paper: *Hydrogen storage and stability of rare earth-doped TiFe alloys under extensive
cycling*. The authors made a single alloy, Ti₁.₀₅Y₀.₀₂Zr₀.₀₃Fe₀.₈Mn₀.₂, and re-measured it after
repeated cycling — so this is one composition, not several. The `Ti` subscript (`1.05`) was a
corrupted date cell, recovered as `1.05`, which matches the formula given in the sheet's note
exactly. `wt` uses the initial (as-activated) capacity. Only ΔH is reported, not plateau
pressure.


In [6]:
block4_raw = dict(Ti=1.05, Y=0.02, Zr=0.03, Fe=0.8, Mn=0.2, wt=1.77)  # Ti recovered from corrupted date cell A40

start = len(block1_rows) + len(block2_rows) + len(block3_rows) + 1
ratio = {el: block4_raw[el] for el in ('Ti', 'Y', 'Zr', 'Fe', 'Mn')}
frac = ratio_to_atomic_fraction(ratio)
block4_rows = [make_row(f'4_{start}', frac, block4_raw['wt'])]

len(block4_rows)


1

### Block 5 — TiFe–C alloys (4 samples)

Paper: *Enhanced initial hydrogenation of TiFe-based hydrogen storage alloys containing C*.
Composition reported in wt%. The `C` values for 3 of the 4 rows were corrupted date cells,
recovered as `2.2`, `4.7`, `6.1` wt% (increasing carbon addition, consistent with the series).
Capacities are approximate, taken from a figure per the sheet's note (prefixed `~` in the
source). The three C-doped rows report an approximate plateau of `~1.0` MPa (abs) / `~0.5` MPa
(des); the baseline (no C) row reports no plateau.


In [7]:
block5_raw = [
    dict(Ti=50.4, Fe=49.6, C=0.0, wt=0.00, p_abs=np.nan, p_des=np.nan),
    dict(Ti=50.8, Fe=48.7, C=2.2, wt=0.80, p_abs=1.0, p_des=0.5),   # C recovered from corrupted date cell C45
    dict(Ti=52.3, Fe=47.2, C=4.7, wt=1.45, p_abs=1.0, p_des=0.5),   # C recovered from corrupted date cell C46
    dict(Ti=53.5, Fe=46.0, C=6.1, wt=1.60, p_abs=1.0, p_des=0.5),   # C recovered from corrupted date cell C47
]

block5_rows = []
start = len(block1_rows) + len(block2_rows) + len(block3_rows) + len(block4_rows) + 1
for i, s in enumerate(block5_raw, start=start):
    wt_pct = {el: s[el] for el in ('Ti', 'Fe', 'C')}
    frac = wt_pct_to_atomic_fraction(wt_pct)
    block5_rows.append(make_row(f'4_{i}', frac, s['wt'], s['p_abs'], s['p_des']))

len(block5_rows)


4

### Block 6 — TiFe doped with Cr (and S) (1 of 2 samples; 1 excluded)

Paper: *Hydrogen Storage in TiFe-Based Alloys Doped with Cr and S: Thermodynamic and Structural
Peculiarities*. Two experimentally prepared alloys: TiFe₀.₉₂Cr₀.₀₈ and TiFe₀.₉Cr₀.₀₈S₀.₀₂.
**`main.csv` has no column for sulfur** (it has Sb, Sc, Si, Sm, Sn — but no plain S), so the
second alloy (which contains S) cannot be represented and is excluded, consistent with how the
mischmetal samples were handled above. The first alloy (no S) is kept.
`wt` was a corrupted date cell, recovered as `1.9`. Only ΔH (two-plateau α↔β / β↔γ
thermodynamics) is reported, not a measured plateau pressure.


In [8]:
block6_raw = [
    dict(Ti=1.00, Fe=0.92, Cr=0.08, wt=1.9),  # wt recovered from corrupted date cell E52
    # excluded: TiFe0.9Cr0.08S0.02 (main.csv has no sulfur column)
]

block6_rows = []
start = len(block1_rows) + len(block2_rows) + len(block3_rows) + len(block4_rows) + len(block5_rows) + 1
for i, s in enumerate(block6_raw, start=start):
    ratio = {el: s[el] for el in ('Ti', 'Fe', 'Cr')}
    frac = ratio_to_atomic_fraction(ratio)
    block6_rows.append(make_row(f'4_{i}', frac, s['wt']))

len(block6_rows)


1

## Merge and validate

In [9]:
new_rows = block1_rows + block2_rows + block3_rows + block4_rows + block5_rows + block6_rows
new_df = pd.DataFrame(new_rows)[main.columns.tolist()]

print(f'New samples: {len(new_df)} (34 candidates in the source file, 4 excluded for undefined mischmetal composition)')

# Validate: every new row's element columns should sum to 1, same as main.csv
sums = new_df[element_cols].sum(axis=1)
assert np.allclose(sums, 1.0), sums[~np.isclose(sums, 1.0)]
print('All new rows sum to 1.0 across element columns: OK')

merged = pd.concat([main, new_df], ignore_index=True)
print('main.csv:', main.shape, '-> main_1.csv:', merged.shape)
assert merged.shape[0] == main.shape[0] + len(new_df)
assert merged.shape[1] == main.shape[1]


New samples: 29 (34 candidates in the source file, 4 excluded for undefined mischmetal composition)
All new rows sum to 1.0 across element columns: OK
main.csv: (179, 62) -> main_1.csv: (208, 62)


In [10]:
merged.to_csv('main_1.csv', index=False)
print('Wrote main_1.csv —', merged.shape[0], 'rows (main.csv itself was not modified).')
new_df


Wrote main_1.csv — 208 rows (main.csv itself was not modified).


,table,wt,P_abs,P_des,Ag,Al,Au,B,Be,Bi,...,Tb,Th,Ti,Tm,U,V,W,Y,Zn,Zr
0,4_1,1.704,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.469569,0.0,0.0,0.000000,0.0,0.008696,0.0,0.043481
1,4_2,1.715,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.459586,0.0,0.0,0.021259,0.0,0.008508,0.0,0.042553
2,4_3,1.732,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.444655,0.0,0.0,0.053044,0.0,0.008237,0.0,0.041173
3,4_4,1.730,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.434745,0.0,0.0,0.074168,0.0,0.008050,0.0,0.040251
4,4_5,1.620,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.419911,0.0,0.0,0.105751,0.0,0.007774,0.0,0.038881
5,4_6,1.770,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.526316,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
6,4_7,1.710,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.500000,0.0,0.0,0.050000,0.0,0.000000,0.0,0.000000
7,4_8,1.790,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.500000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
8,4_9,1.670,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.500000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
9,4_10,1.360,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.500000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000
